In [2]:
import pandas as pd

movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data//ratings.csv")

print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [6]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [5]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [7]:
ratings.userId.nunique()

610

In [8]:
ratings.movieId.nunique()

9724

In [9]:
movie_ratings = ratings.merge(
    movies,
    on="movieId",
    how="left"
)

movie_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [10]:
movie_stats = (
    movie_ratings
    .groupby("title")
    .agg(
        rating_count=("rating", "count"),
        avg_rating=("rating", "mean")
    )
    .reset_index()
)

movie_stats.sort_values(
    by="rating_count",
    ascending=False
).head(20)

,title,rating_count,avg_rating
3158,Forrest Gump (1994),329,4.164134
7593,"Shawshank Redemption, The (1994)",317,4.429022
6865,Pulp Fiction (1994),307,4.197068
7680,"Silence of the Lambs, The (1991)",279,4.161290
5512,"Matrix, The (1999)",278,4.192446
8001,Star Wars: Episode IV - A New Hope (1977),251,4.231076
4662,Jurassic Park (1993),238,3.750000
1337,Braveheart (1995),237,4.031646
8363,Terminator 2: Judgment Day (1991),224,3.970982
7421,Schindler's List (1993),220,4.225000


In [11]:
def recommend_popular_movies(movie_stats, n=10):
    return (
        movie_stats
        .sort_values(
            by="rating_count",
            ascending=False
        )
        .head(n)
    )

In [12]:
recommend_popular_movies(movie_stats)

,title,rating_count,avg_rating
3158,Forrest Gump (1994),329,4.164134
7593,"Shawshank Redemption, The (1994)",317,4.429022
6865,Pulp Fiction (1994),307,4.197068
7680,"Silence of the Lambs, The (1991)",279,4.161290
5512,"Matrix, The (1999)",278,4.192446
8001,Star Wars: Episode IV - A New Hope (1977),251,4.231076
4662,Jurassic Park (1993),238,3.750000
1337,Braveheart (1995),237,4.031646
8363,Terminator 2: Judgment Day (1991),224,3.970982
7421,Schindler's List (1993),220,4.225000


In [13]:
user_movie_matrix = movie_ratings.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

user_movie_matrix.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
sparsity = 1 - (
    ratings.shape[0] /
    (ratings["userId"].nunique()
     * movies["movieId"].nunique())
)

print(sparsity)

0.9830317267467884


In [15]:
user_movie_filled = user_movie_matrix.fillna(0)

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
user_similarity = cosine_similarity(
    user_movie_filled
)

In [19]:

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_filled.index,
    columns=user_movie_filled.index
)

user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


In [21]:
user_similarity_df.loc[1].sort_values(ascending=False).head(10)

userId
1      1.000000
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
469    0.330664
39     0.329782
288    0.329700
452    0.328048
Name: 1, dtype: float64

In [22]:
user_id = 1

similar_users = (
    user_similarity_df[user_id]
    .sort_values(ascending=False)
    .drop(user_id)
)

similar_users.head(10)

userId
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
469    0.330664
39     0.329782
288    0.329700
452    0.328048
45     0.327922
Name: 1, dtype: float64

In [23]:
most_similar_user = similar_users.index[0]
print(most_similar_user)

266


In [30]:
user1_movies = set(
    ratings[ratings["userId"] == 1]["movieId"]
)

In [31]:
neighbor_movies = set(
    ratings[
        ratings["userId"] == most_similar_user
    ]["movieId"]
)

In [32]:
recommendations = neighbor_movies - user1_movies

len(recommendations)

113

In [33]:
recommended_movies = movies[
    movies["movieId"].isin(recommendations)
]

recommended_movies[["movieId", "title"]].head(10)

,movieId,title
15,16,Casino (1995)
16,17,Sense and Sensibility (1995)
20,21,Get Shorty (1995)
23,24,Powder (1995)
31,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
35,39,Clueless (1995)
41,45,To Die For (1995)
57,64,Two if by Sea (1996)
61,69,Friday (1995)
84,95,Broken Arrow (1996)


In [34]:
movie_user_matrix = movie_ratings.pivot_table(
    index="title",
    columns="userId",
    values="rating"
).fillna(0)

movie_user_matrix.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Round Midnight (1986),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Salem's Lot (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Til There Was You (1997),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [35]:
movie_similarity = cosine_similarity(
    movie_user_matrix
)

In [36]:
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_user_matrix.index,
    columns=movie_user_matrix.index
)

In [40]:
movie_name = "Batman (1989)"
movies[
    movies["title"].str.contains(
        "Batman",
        case=False,
        na=False
    )
]["title"]

126                               Batman Forever (1995)
509                                       Batman (1989)
1060                              Batman Returns (1992)
1174                              Batman & Robin (1997)
2418                Batman: Mask of the Phantasm (1993)
5463                                      Batman (1966)
5620                  Batman/Superman Movie, The (1998)
5631          Batman Beyond: Return of the Joker (2000)
5917                               Batman Begins (2005)
6815                       Batman: Gotham Knight (2008)
7380                  Batman: Under the Red Hood (2010)
7731                            Batman: Year One (2011)
7903             Superman/Batman: Public Enemies (2009)
8032     Batman: The Dark Knight Returns, Part 1 (2012)
8080     Batman: The Dark Knight Returns, Part 2 (2013)
8195    LEGO Batman: The Movie - DC Heroes Unite (2013)
8234             Batman: Mystery of the Batwoman (2003)
8486                   Batman: Assault on Arkham

In [41]:
movie_similarity_df[movie_name] \
    .sort_values(
        ascending=False
    ) \
    .head(11)

title
Batman (1989)                        1.000000
Batman Forever (1995)                0.705639
True Lies (1994)                     0.696755
Terminator 2: Judgment Day (1991)    0.645603
Fugitive, The (1993)                 0.639785
Jurassic Park (1993)                 0.639496
Dances with Wolves (1990)            0.599986
Stargate (1994)                      0.599201
Mask, The (1994)                     0.597826
Aladdin (1992)                       0.596721
Braveheart (1995)                    0.591909
Name: Batman (1989), dtype: float64